# Chapter 14 - Learning to Write Like Shakespeare: Long Short-Term Memory

## Summary
Plain RNNs forget over long ranges because gradients **vanish** (or explode). The **LSTM** fixes this with a *separate memory cell* and *gates* that control information flow.
- **Cell state `c_t`**: a conveyor belt of memory; it is updated additively, so gradients flow through it far more easily than through repeated matrix multiplications.
- **Three gates** (each a sigmoid layer, outputting 0-1 per unit):
  - **Forget gate `f`**: how much of the old memory to keep.
  - **Input gate `i`**: how much of the new candidate `g` to write.
  - **Output gate `o`**: how much of the (squashed) memory to expose as hidden state.
- **Character-level language model:** train the LSTM to predict the next character, then sample from it to generate text.
- **Truncated BPTT:** train on windows of `bptt` characters, carrying nothing else but the weights between windows (here the hidden state restarts each window).
- **Tips from the book:** sampling with *temperature*, and a *regularised, well-initialised* network trained for many steps give better text.

## Theory
Given input `x_t` and previous states `h_{t-1}, c_{t-1}`:

```
f = sigmoid(W_xf x + W_hf h_prev)        # forget gate
i = sigmoid(W_xi x + W_hi h_prev)        # input gate
o = sigmoid(W_xo x + W_ho h_prev)        # output gate
g = tanh   (W_xg x + W_hg h_prev)        # candidate memory
c = f * c_prev + i * g                   # new cell state (additive update!)
h = o * tanh(c)                          # new hidden state
```

**Why it helps:** `dc_t/dc_{t-1} = f`. If the network keeps `f` near 1 the gradient passes through unchanged - the "constant error carousel". The gates learn *when* to remember, write and read, so the model can track long-range structure (e.g. closing a quote opened many characters ago).

**Sampling temperature:** divide the logits by `T` before softmax. Low `T` -> conservative and repetitive, high `T` -> diverse but error-prone.

The model below is built from the autograd framework introduced in Chapter 13 (included again so the notebook runs on its own).

In [1]:
# If packages are missing in this kernel, uncomment:
# %pip install numpy scikit-learn matplotlib
import numpy as np

In [2]:
# ---------- Mini deep-learning framework (autograd Tensor + layers + optimizer) ----------
import numpy as np

def _acc(t, g):
    # accumulate gradient g into tensor t (only if it tracks gradients)
    if t.autograd:
        t.grad = g if t.grad is None else t.grad + g

class Tensor:
    def __init__(self, data, autograd=False, creators=None, op=None):
        self.data = np.asarray(data, dtype=float)
        self.autograd, self.creators, self.op = autograd, creators, op
        self.grad, self.extra = None, {}

    def _new(self, data, creators, op, **kw):
        ag = any(c.autograd for c in creators)
        t = Tensor(data, autograd=ag, creators=creators if ag else None, op=op)
        t.extra = kw
        return t

    # ----- operations (each records how to send gradients back) -----
    def __add__(self, o): return self._new(self.data + o.data, [self, o], "add")
    def __sub__(self, o): return self._new(self.data - o.data, [self, o], "sub")
    def __mul__(self, o): return self._new(self.data * o.data, [self, o], "mul")
    def __neg__(self):    return self._new(-self.data, [self], "neg")
    def sum(self, dim):   return self._new(self.data.sum(dim), [self], "sum", dim=dim)
    def expand(self, dim, copies):
        data = np.repeat(np.expand_dims(self.data, dim), copies, axis=dim)
        return self._new(data, [self], "expand", dim=dim)
    def transpose(self):  return self._new(self.data.T, [self], "transpose")
    def mm(self, o):      return self._new(self.data.dot(o.data), [self, o], "mm")
    def sigmoid(self):    return self._new(1 / (1 + np.exp(-self.data)), [self], "sigmoid")
    def tanh(self):       return self._new(np.tanh(self.data), [self], "tanh")
    def relu(self):       return self._new(np.maximum(self.data, 0), [self], "relu")
    def index_select(self, idx):
        idx = np.asarray(idx.data if isinstance(idx, Tensor) else idx, dtype=int)
        return self._new(self.data[idx], [self], "index_select", idx=idx)
    def cross_entropy(self, target):
        t = np.asarray(target.data if isinstance(target, Tensor) else target, dtype=int)
        e = np.exp(self.data - self.data.max(1, keepdims=True))
        soft = e / e.sum(1, keepdims=True)
        loss = -np.log(soft[np.arange(len(t)), t] + 1e-12).mean()
        return self._new(loss, [self], "cross_entropy", soft=soft, t=t)

    # ----- backpropagation: topological order, then local gradient rules -----
    def backward(self, grad=None):
        order, seen = [], set()
        def visit(t):
            if id(t) in seen or not t.autograd: return
            seen.add(id(t))
            for c in (t.creators or []): visit(c)
            order.append(t)
        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else grad
        for t in reversed(order):
            if t.creators is not None: t._propagate()

    def _propagate(self):
        g, c, op = self.grad, self.creators, self.op
        if op == "add":   _acc(c[0], g); _acc(c[1], g)
        elif op == "sub": _acc(c[0], g); _acc(c[1], -g)
        elif op == "mul": _acc(c[0], g * c[1].data); _acc(c[1], g * c[0].data)
        elif op == "neg": _acc(c[0], -g)
        elif op == "sum": _acc(c[0], np.broadcast_to(np.expand_dims(g, self.extra["dim"]), c[0].data.shape).copy())
        elif op == "expand": _acc(c[0], g.sum(self.extra["dim"]))
        elif op == "transpose": _acc(c[0], g.T)
        elif op == "mm":  _acc(c[0], g.dot(c[1].data.T)); _acc(c[1], c[0].data.T.dot(g))
        elif op == "sigmoid": _acc(c[0], g * self.data * (1 - self.data))
        elif op == "tanh":    _acc(c[0], g * (1 - self.data ** 2))
        elif op == "relu":    _acc(c[0], g * (c[0].data > 0))
        elif op == "index_select":
            gr = np.zeros_like(c[0].data); np.add.at(gr, self.extra["idx"], g); _acc(c[0], gr)
        elif op == "cross_entropy":
            d = self.extra["soft"].copy(); d[np.arange(len(d)), self.extra["t"]] -= 1
            _acc(c[0], d / len(d) * g)

    def __repr__(self): return f"Tensor({self.data})"

# ---------- Layers ----------
class Layer:
    def __init__(self): self.parameters = []
    def get_parameters(self): return self.parameters

class Linear(Layer):
    def __init__(self, n_in, n_out, bias=True):
        super().__init__()
        self.weight = Tensor(np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in), autograd=True)
        self.bias = Tensor(np.zeros(n_out), autograd=True) if bias else None
        self.parameters = [self.weight] + ([self.bias] if bias else [])
    def forward(self, x):
        out = x.mm(self.weight)
        return out + self.bias.expand(0, len(x.data)) if self.bias is not None else out

class Sequential(Layer):
    def __init__(self, layers): super().__init__(); self.layers = layers
    def forward(self, x):
        for l in self.layers: x = l.forward(x)
        return x
    def get_parameters(self): return [p for l in self.layers for p in l.get_parameters()]

class Tanh(Layer):
    def forward(self, x): return x.tanh()
class Sigmoid(Layer):
    def forward(self, x): return x.sigmoid()
class ReLU(Layer):
    def forward(self, x): return x.relu()

class Embedding(Layer):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.weight = Tensor((np.random.rand(vocab_size, dim) - 0.5) / dim, autograd=True)
        self.parameters = [self.weight]
    def forward(self, idx): return self.weight.index_select(idx)

class MSELoss(Layer):
    def forward(self, pred, target): return ((pred - target) * (pred - target)).sum(0)
class CrossEntropyLoss(Layer):
    def forward(self, pred, target_idx): return pred.cross_entropy(target_idx)

class SGD:
    def __init__(self, parameters, alpha=0.1): self.parameters, self.alpha = parameters, alpha
    def step(self, zero=True):
        for p in self.parameters:
            if p.grad is not None:
                p.data -= p.grad * self.alpha
                if zero: p.grad = None

### 1) LSTM cell, built from framework layers

In [3]:
class LSTMCell(Layer):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.n_hidden = n_hidden
        self.xf, self.xi, self.xo, self.xc = [Linear(n_in, n_hidden) for _ in range(4)]
        self.hf, self.hi, self.ho, self.hc = [Linear(n_hidden, n_hidden, bias=False) for _ in range(4)]
        self.w_ho = Linear(n_hidden, n_out)
        for l in (self.xf, self.xi, self.xo, self.xc, self.hf, self.hi, self.ho, self.hc, self.w_ho):
            self.parameters += l.get_parameters()
        self.xf.bias.data += 1.0          # start with forget gate open (remember by default)

    def forward(self, x, hidden):
        prev_h, prev_c = hidden
        f = (self.xf.forward(x) + self.hf.forward(prev_h)).sigmoid()
        i = (self.xi.forward(x) + self.hi.forward(prev_h)).sigmoid()
        o = (self.xo.forward(x) + self.ho.forward(prev_h)).sigmoid()
        g = (self.xc.forward(x) + self.hc.forward(prev_h)).tanh()
        c = (f * prev_c) + (i * g)
        h = o * c.tanh()
        return self.w_ho.forward(h), (h, c)

    def init_hidden(self, batch_size=1):
        z = lambda: Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)
        return (z(), z())
print("LSTMCell defined")

LSTMCell defined


### 2) Data: a Shakespeare snippet as characters

In [4]:
text = ("to be, or not to be, that is the question:\n"
        "whether 'tis nobler in the mind to suffer\n"
        "the slings and arrows of outrageous fortune,\n"
        "or to take arms against a sea of troubles\n"
        "and by opposing end them. to die, to sleep,\n"
        "no more; and by a sleep to say we end\n"
        "the heart-ache and the thousand natural shocks\n"
        "that flesh is heir to: 'tis a consummation\n"
        "devoutly to be wish'd.\n")
chars = sorted(set(text)); V = len(chars)
c2i = {c: i for i, c in enumerate(chars)}; i2c = {i: c for c, i in c2i.items()}
data = np.array([c2i[c] for c in text])
print("characters:", len(data), "| vocab size:", V)

characters: 367 | vocab size: 31


### 3) Train with truncated BPTT (windows of 16 characters, batch of 8)

In [5]:
np.random.seed(3)
embed = Embedding(V, 24)
cell = LSTMCell(24, 64, V)
crit = CrossEntropyLoss()
params = embed.get_parameters() + cell.get_parameters()
optim = SGD(params, alpha=0.5)
bptt, batch = 16, 8

def clip(params, c=5.0):
    for p in params:
        if p.grad is not None: np.clip(p.grad, -c, c, out=p.grad)

losses = []
for it in range(401):
    starts = np.random.randint(0, len(data) - bptt - 1, batch)
    h = cell.init_hidden(batch); total = None
    for t in range(bptt):
        x = data[starts + t]; y = data[starts + t + 1]
        out, h = cell.forward(embed.forward(x), h)
        l = crit.forward(out, y)
        total = l if total is None else total + l
    total.backward()
    clip(params); optim.step()
    losses.append(float(total.data) / bptt)
    if it % 100 == 0: print(f"iter {it:3d}  loss/char {losses[-1]:.3f}")

iter   0  loss/char 3.432
iter 100  loss/char 1.299
iter 200  loss/char 0.731
iter 300  loss/char 0.451
iter 400  loss/char 0.502


### 4) Generate text

In [6]:
def generate(seed="to", n=200, temperature=0.7):
    h = cell.init_hidden(1); out_text = seed
    for ch in seed[:-1]:
        _, h = cell.forward(embed.forward(np.array([c2i[ch]])), h)
    ch = seed[-1]
    for _ in range(n):
        out, h = cell.forward(embed.forward(np.array([c2i[ch]])), h)
        logits = out.data[0] / temperature
        p = np.exp(logits - logits.max()); p /= p.sum()
        ch = i2c[np.random.choice(V, p=p)]
        out_text += ch
    return out_text
np.random.seed(5)
print(generate("to", 220, 0.6))
print("-----")
print(generate("the", 120, 1.0))

to say we end
the heir to say we end
the to more; and arrows of outrageous fortune,
or to: 'tis a sea of troubles
and natural shocks
that flesh is to be, that is to sleep to: 'tis a consummation
devoutly to be wish is to b
-----
the question
be wishocks
that flesh is more; and by opposing end that is that is to s flesh isleep to say we end
the questi


## Takeaways
- The LSTM's **additive cell state and gates** let gradients and information persist over long spans, fixing the main weakness of plain RNNs.
- Because the framework provides autograd, the whole LSTM is ~15 lines of forward code - no hand-written backward pass.
- Generated text improves with more data, more units and longer training; here we use a tiny corpus so it runs in seconds.